In [ ]:
from tiledpy.tileset import decode_gid


def get_spawn_points(tmap, layer_name="suelo", scale=4):
    """Devuelve lista de posiciones pixel (x, y) de tiles con 'player spawn'."""
    layer = tmap.get_layer(layer_name)
    if layer is None:
        return []

    spawn_points = []
    for tx, ty, raw_gid in layer.iter_tiles():
        gid, _ = decode_gid(raw_gid)
        if gid == 0:
            continue

        tileset = tmap.get_tileset_for_gid(gid)
        if tileset is None:
            continue

        local_id = tileset.global_to_local(gid)
        tile_data = tileset.tile_data.get(local_id)

        if tile_data:
            print(tile_data.properties)
            
        if tile_data and tile_data.properties.get("player_spawn"):
            px = tx * tmap.tile_width * scale
            py = ty * tmap.tile_height * scale
            spawn_points.append((px, py))

    return spawn_points


In [ ]:
import pygame
from tiledpy import TiledMap

pygame.init()
screen = pygame.display.set_mode((800, 600))
clock  = pygame.time.Clock()

tmap = TiledMap(r"C:\Users\sergi\Downloads\TileMap\demo.tmx")
cam_x, cam_y = 1000, 1000

running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    if running:
        screen.fill((0, 0, 0))
        tmap.draw_all_layers(screen, offset=(cam_x, cam_y), scale = 4)
        pygame.display.flip()
        clock.tick(60)

pygame.quit()

spawns = get_spawn_points(tmap, layer_name="objetos", scale=4)
player_x, player_y = spawns[0]  # o random.choice(spawns)

player_x, player_y

In [1]:
import pygame
from tiledpy import TiledMap
from tiledpy.tileset import decode_gid

pygame.init()
screen = pygame.display.set_mode((800, 600))
clock  = pygame.time.Clock()

tmap   = TiledMap(r"C:\Users\sergi\Downloads\TileMap\demo.tmx")
SCALE  = 4
cam_x, cam_y = 1000, 1000

COLLISION_COLOR = (173, 216, 230, 120)  # light blue, semitransparente

def draw_collisions(surface, tmap, offset, scale):
    ox, oy = offset
    overlay = pygame.Surface(surface.get_size(), pygame.SRCALPHA)

    for layer in tmap.get_tile_layers():
        if not layer.visible:
            continue
        for tx, ty, raw_gid in layer.iter_tiles():
            gid = raw_gid
            if gid == 0:
                continue
            ts = tmap.get_tileset_for_gid(gid)
            if ts is None:
                continue
            tile_data = ts.tile_data.get(ts.global_to_local(gid))
            if not tile_data or not tile_data.collision_objects:
                continue

            # Origen del tile en pantalla
            tile_px = tx * tmap.tile_width  * scale - ox
            tile_py = ty * tmap.tile_height * scale - oy

            for col in tile_data.collision_objects:
                rect = pygame.Rect(
                    tile_px + col["x"]    * scale,
                    tile_py + col["y"]    * scale,
                    col["width"]          * scale,
                    col["height"]         * scale,
                )
                pygame.draw.rect(overlay, COLLISION_COLOR, rect)
                pygame.draw.rect(overlay, (100, 180, 255, 200), rect, 1)  # borde

    surface.blit(overlay, (0, 0))


running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    if running:
        screen.fill((0, 0, 0))
        tmap.draw_all_layers(screen, offset=(cam_x, cam_y), scale=SCALE)
        draw_collisions(screen, tmap, offset=(cam_x, cam_y), scale=SCALE)
        pygame.display.flip()
        clock.tick(60)

pygame.quit()


c:\Users\sergi\anaconda3\envs\PYGAME\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
from tiledpy import TiledMap
from tiledpy.tileset import decode_gid

tmap = TiledMap(r"C:\Users\sergi\Downloads\TileMap\demo.tmx")

tiles = []
for layer in tmap.visible_layers:
    tiles.extend( layer.get_tile_by_property("Class", "spawn") )

decode_gid(tmap.get_tile_gid(0, 0))

(5, TileFlags(flip_h=False, flip_v=False, flip_d=False))

In [8]:
import time

def count():
    print("One")
    time.sleep(1)
    print("Two")
    time.sleep(1)

def main():
    for _ in range(3):
        count()



start = time.perf_counter()
main()
elapsed = time.perf_counter() - start
print(f"executed in {elapsed:0.2f} seconds.")

One
Two
One
Two
One
Two
executed in 6.00 seconds.


In [9]:
import asyncio

async def count():
    print("One")
    await asyncio.sleep(1)
    print("Two")
    await asyncio.sleep(1)

async def main():
    await asyncio.gather(count(), count(), count())


import time

start = time.perf_counter()
await main()
elapsed = time.perf_counter() - start
print(f"executed in {elapsed:0.2f} seconds.")

One
One
One
Two
Two
Two
executed in 2.01 seconds.
